# 03. 데이터 전처리 및 병합

**프로젝트**: 창원시 폭우·기후재난 AI 선제 대응 시스템  
**목적**: 원본 데이터를 정제·변환하고, 공간 조인으로 통합 분석 데이터셋을 생성

---

## 1. 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

print('전처리 환경 준비 완료')

## 2. 데이터 로드 및 컬럼 정리

In [ ]:
# TODO: 각 데이터 로드 및 필요 컬럼만 선택

# --- 하수관로 ---
# df_sewer = pd.read_csv(RAW_DIR / 'sewer_pipe_status.csv', encoding='cp949')
# df_sewer = df_sewer[df_sewer['시군구'].str.contains('창원')]
# df_sewer = df_sewer[['시군구', '읍면동', '관종', '관경', '연장', '매설연도']].copy()
# df_sewer['노후연수'] = 2026 - df_sewer['매설연도']
# df_sewer['노후여부'] = df_sewer['노후연수'] >= 30

# --- 강수량 ---
# df_rainfall = pd.read_csv(RAW_DIR / 'hourly_rainfall.csv', encoding='cp949')
# df_rainfall['datetime'] = pd.to_datetime(df_rainfall['일시'])
# df_rainfall['rainfall_mm'] = pd.to_numeric(df_rainfall['강수량(mm)'], errors='coerce')

# --- 배수펌프장 ---
# df_pump = pd.read_csv(RAW_DIR / 'drainage_pump.csv', encoding='cp949')
# df_pump = df_pump[df_pump['시군구'].str.contains('창원')]

# --- 건축물대장 ---
# df_building = pd.read_csv(RAW_DIR / 'building_register.csv', encoding='cp949')
# df_building['건축연도'] = pd.to_numeric(df_building['사용승인일자'].str[:4], errors='coerce')
# df_building['건물나이'] = 2026 - df_building['건축연도']

# --- 인구통계 ---
# df_pop = pd.read_csv(RAW_DIR / 'population_by_dong.csv', encoding='cp949')

print('데이터 로드 후 주석 해제')

## 3. 결측치 처리

In [ ]:
def handle_missing(df, name, strategy='drop'):
    """결측치 처리 및 결과 보고"""
    before = len(df)
    missing_cols = df.columns[df.isnull().any()].tolist()
    
    if not missing_cols:
        print(f'{name}: 결측치 없음 ✅')
        return df
    
    print(f'{name}: 결측 컬럼 {missing_cols}')
    
    if strategy == 'drop':
        df = df.dropna(subset=missing_cols)
    elif strategy == 'fill_median':
        for col in missing_cols:
            if df[col].dtype in ['float64', 'int64']:
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna('미상')
    
    after = len(df)
    print(f'  {before:,} → {after:,}행 ({before - after:,}행 제거)')
    return df

# TODO: 각 데이터에 적용
# df_sewer = handle_missing(df_sewer, '하수관로', 'fill_median')
# df_rainfall = handle_missing(df_rainfall, '강수량', 'drop')

## 4. 좌표 변환

모든 공간 데이터를 동일한 좌표계(WGS84, EPSG:4326)로 통일합니다.

In [ ]:
def to_geodataframe(df, lon_col, lat_col, crs='EPSG:4326'):
    """DataFrame → GeoDataFrame 변환"""
    geometry = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)
    return gdf

def convert_tm_to_wgs84(df, x_col, y_col):
    """TM 좌표 → WGS84 변환 (공공데이터에서 흔한 케이스)"""
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[x_col], df[y_col]),
        crs='EPSG:5186'  # Korea TM Central Belt
    )
    gdf = gdf.to_crs('EPSG:4326')
    df['경도'] = gdf.geometry.x
    df['위도'] = gdf.geometry.y
    return df

# TODO: 좌표 데이터가 있는 데이터셋에 적용
# df_pump = convert_tm_to_wgs84(df_pump, 'X좌표', 'Y좌표')
print('좌표 변환 함수 준비 완료')

## 5. 분석 격자 생성

창원시를 500m × 500m 격자로 분할하여 격자별 위험도를 산출합니다.

In [ ]:
def create_grid(bounds, cell_size=0.005):
    """
    분석 격자 생성
    bounds: (min_lon, min_lat, max_lon, max_lat)
    cell_size: 격자 크기 (약 0.005도 ≈ 500m)
    """
    from shapely.geometry import box
    
    min_lon, min_lat, max_lon, max_lat = bounds
    grid_cells = []
    grid_ids = []
    idx = 0
    
    x = min_lon
    while x < max_lon:
        y = min_lat
        while y < max_lat:
            cell = box(x, y, x + cell_size, y + cell_size)
            grid_cells.append(cell)
            grid_ids.append(f'G{idx:04d}')
            idx += 1
            y += cell_size
        x += cell_size
    
    grid = gpd.GeoDataFrame(
        {'grid_id': grid_ids},
        geometry=grid_cells,
        crs='EPSG:4326'
    )
    print(f'격자 {len(grid)}개 생성 ({cell_size * 111:.0f}m × {cell_size * 111:.0f}m)')
    return grid

# 창원시 대략적 경계 (min_lon, min_lat, max_lon, max_lat)
CHANGWON_BOUNDS = (128.45, 35.05, 128.85, 35.35)

# TODO: 격자 생성
# grid = create_grid(CHANGWON_BOUNDS, cell_size=0.005)
# print(grid.head())
print('격자 생성 함수 준비 완료')

## 6. 공간 조인 — 격자별 데이터 집계

In [ ]:
# TODO: 각 데이터를 격자에 공간 조인

# --- 격자별 노후 관로 비율 ---
# gdf_sewer = to_geodataframe(df_sewer, '경도', '위도')
# joined = gpd.sjoin(grid, gdf_sewer, how='left', predicate='contains')
# grid['노후관로비율'] = joined.groupby('grid_id')['노후여부'].mean()
# grid['관로총연장'] = joined.groupby('grid_id')['연장'].sum()

# --- 격자별 배수펌프 용량 ---
# gdf_pump = to_geodataframe(df_pump, '경도', '위도')
# joined_pump = gpd.sjoin(grid, gdf_pump, how='left', predicate='contains')
# grid['펌프용량합계'] = joined_pump.groupby('grid_id')['용량'].sum().fillna(0)

# --- 격자별 노후건물 비율 ---
# gdf_building = to_geodataframe(df_building, '경도', '위도')
# joined_bld = gpd.sjoin(grid, gdf_building, how='left', predicate='contains')
# grid['노후건물비율'] = joined_bld.groupby('grid_id')['건물나이'].apply(
#     lambda x: (x >= 30).mean()
# )

print('공간 조인 코드 준비 완료 — 데이터 로드 후 실행')

## 7. 통합 데이터셋 저장

In [ ]:
# TODO: 처리된 데이터 저장

# grid.to_file(PROCESSED_DIR / 'analysis_grid.geojson', driver='GeoJSON')
# df_rainfall_clean.to_csv(PROCESSED_DIR / 'rainfall_cleaned.csv', index=False)
# df_sewer_clean.to_csv(PROCESSED_DIR / 'sewer_cleaned.csv', index=False)

# print('=== 저장 완료 ===')
# print(f'  분석 격자: {PROCESSED_DIR / "analysis_grid.geojson"}')
# print(f'  강수량: {PROCESSED_DIR / "rainfall_cleaned.csv"}')
# print(f'  하수관로: {PROCESSED_DIR / "sewer_cleaned.csv"}')

print('전처리 완료 후 주석 해제하여 저장')